# CouWang Coupon Analysis
쿠왕(CouWang) 프로젝트용 CSV 기반 분석 노트북입니다.

이 노트북은 다음을 포함합니다.
1. 데이터 로드
2. 기본 분포 확인
3. 퍼널 분석
4. 핵심 KPI 요약
5. 알림 효과 비교
6. 브랜드별 만료율 분석
7. D1 / D7 리텐션 느낌 분석
8. 일별 등록 추이
9. 핵심 인사이트 정리

## 1. 데이터 로드

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# 차트 한글 깨짐 방지(환경에 맞게 필요 시 변경)
plt.rcParams["font.family"] = "AppleGothic"   # macOS
# plt.rcParams["font.family"] = "Malgun Gothic"  # Windows
plt.rcParams["axes.unicode_minus"] = False

# 프로젝트 루트 탐색
cwd = Path.cwd()

if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("data 폴더를 찾을 수 없습니다. analysis 폴더 또는 프로젝트 루트에서 실행하세요.")

DATA_DIR = PROJECT_ROOT / "data"
COUPONS_PATH = DATA_DIR / "coupons.csv"
EVENTS_PATH = DATA_DIR / "events.csv"

coupons = pd.read_csv(COUPONS_PATH)
events = pd.read_csv(EVENTS_PATH)

# 날짜형 컬럼 파싱
coupons["created_at"] = pd.to_datetime(coupons["created_at"], errors="coerce")
coupons["expiry_date"] = pd.to_datetime(coupons["expiry_date"], errors="coerce")
coupons["redeemed_at"] = pd.to_datetime(coupons["redeemed_at"], errors="coerce")

events["timestamp"] = pd.to_datetime(events["timestamp"], errors="coerce")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("coupons shape:", coupons.shape)
print("events shape:", events.shape)

print("\n[coupons preview]")
display(coupons.head())

print("\n[events preview]")
display(events.head())

## 2. 기본 분포 확인

In [ ]:
print("쿠폰 상태 분포")
display(coupons["status"].value_counts(dropna=False).rename_axis("status").reset_index(name="count"))

print("\n이벤트 분포")
display(events["event_name"].value_counts(dropna=False).rename_axis("event_name").reset_index(name="count"))

## 3. 퍼널 분석

In [ ]:
funnel_steps = [
    "coupon__start__create",
    "coupon__complete__create",
    "coupon__view__detail",
    "coupon__complete__redeem",
]

user_funnel = (
    events[events["event_name"].isin(funnel_steps)]
    .groupby("event_name")["user_id"]
    .nunique()
    .reindex(funnel_steps)
    .reset_index()
)

user_funnel.columns = ["step", "users"]
user_funnel["conversion_from_start_pct"] = (user_funnel["users"] / user_funnel.loc[0, "users"] * 100).round(2)

display(user_funnel)

In [ ]:
coupon_funnel = (
    events[events["event_name"].isin(funnel_steps)]
    .groupby("event_name")["coupon_id"]
    .nunique()
    .reindex(funnel_steps)
    .reset_index()
)

coupon_funnel.columns = ["step", "coupons"]
coupon_funnel["conversion_from_start_pct"] = (coupon_funnel["coupons"] / coupon_funnel.loc[0, "coupons"] * 100).round(2)

display(coupon_funnel)

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(user_funnel["step"], user_funnel["users"])
plt.title("쿠왕 퍼널(유저 기준)")
plt.xlabel("단계")
plt.ylabel("유저 수")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 4. 핵심 KPI 요약

In [ ]:
total_coupons = len(coupons)
redeemed_coupons = (coupons["status"] == "redeemed").sum()
expired_coupons = (coupons["status"] == "expired").sum()
active_coupons = (coupons["status"] == "active").sum()

redemption_rate = round(redeemed_coupons / total_coupons * 100, 2)
expiration_rate = round(expired_coupons / total_coupons * 100, 2)
active_rate = round(active_coupons / total_coupons * 100, 2)

summary_kpi = pd.DataFrame({
    "metric": ["전체 쿠폰 수", "사용 완료 쿠폰 수", "만료 쿠폰 수", "사용 가능 쿠폰 수", "사용 완료율", "만료율", "사용 가능 비율"],
    "value": [total_coupons, redeemed_coupons, expired_coupons, active_coupons, f"{redemption_rate}%", f"{expiration_rate}%", f"{active_rate}%"]
})

display(summary_kpi)

In [ ]:
status_counts = coupons["status"].value_counts().reindex(["redeemed", "expired", "active"]).fillna(0)

plt.figure(figsize=(6, 4))
plt.bar(status_counts.index, status_counts.values)
plt.title("쿠폰 상태 분포")
plt.xlabel("상태")
plt.ylabel("쿠폰 수")
plt.tight_layout()
plt.show()

## 5. 알림 효과 비교

In [ ]:
users_sent = set(events.loc[events["event_name"] == "notification__send__expiry_reminder", "user_id"].unique())
users_opened = set(events.loc[events["event_name"] == "notification__open__expiry_reminder", "user_id"].unique())

notification_users = pd.DataFrame({"user_id": list(users_sent)})
notification_users["notification_group"] = notification_users["user_id"].apply(
    lambda x: "opened" if x in users_opened else "not_opened"
)

display(notification_users.head())
print(notification_users["notification_group"].value_counts())

In [ ]:
user_redeem_flag = (
    coupons.groupby("user_id")["status"]
    .apply(lambda s: int((s == "redeemed").any()))
    .reset_index(name="has_redeemed_coupon")
)

notification_user_redeem = notification_users.merge(user_redeem_flag, on="user_id", how="left")
notification_user_redeem["has_redeemed_coupon"] = notification_user_redeem["has_redeemed_coupon"].fillna(0)

user_open_effect = (
    notification_user_redeem.groupby("notification_group")
    .agg(
        users=("user_id", "nunique"),
        users_with_redeem=("has_redeemed_coupon", "sum")
    )
    .reset_index()
)

user_open_effect["redeem_user_rate_pct"] = (
    user_open_effect["users_with_redeem"] / user_open_effect["users"] * 100
).round(2)

display(user_open_effect)

In [ ]:
coupon_open_effect = (
    notification_users.merge(coupons[["user_id", "coupon_id", "status"]], on="user_id", how="inner")
    .groupby("notification_group")
    .agg(
        total_coupons=("coupon_id", "count"),
        redeemed_coupons=("status", lambda s: (s == "redeemed").sum())
    )
    .reset_index()
)

coupon_open_effect["coupon_redemption_rate_pct"] = (
    coupon_open_effect["redeemed_coupons"] / coupon_open_effect["total_coupons"] * 100
).round(2)

display(coupon_open_effect)

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(coupon_open_effect["notification_group"], coupon_open_effect["coupon_redemption_rate_pct"])
plt.title("알림 오픈 그룹 vs 미오픈 그룹 사용 완료율")
plt.xlabel("알림 그룹")
plt.ylabel("사용 완료율(%)")
plt.tight_layout()
plt.show()

## 6. 브랜드별 만료율 Top5

In [ ]:
brand_stats = (
    coupons.groupby("brand")
    .agg(
        total_coupons=("coupon_id", "count"),
        expired_coupons=("status", lambda s: (s == "expired").sum()),
        redeemed_coupons=("status", lambda s: (s == "redeemed").sum())
    )
    .reset_index()
)

brand_stats["expiration_rate_pct"] = (brand_stats["expired_coupons"] / brand_stats["total_coupons"] * 100).round(2)
brand_stats["redemption_rate_pct"] = (brand_stats["redeemed_coupons"] / brand_stats["total_coupons"] * 100).round(2)

brand_top5_expire = (
    brand_stats[brand_stats["total_coupons"] >= 5]
    .sort_values(["expiration_rate_pct", "total_coupons"], ascending=[False, False])
    .head(5)
)

display(brand_top5_expire)

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(brand_top5_expire["brand"], brand_top5_expire["expiration_rate_pct"])
plt.title("브랜드별 만료율 Top5")
plt.xlabel("브랜드")
plt.ylabel("만료율(%)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 7. D1 / D7 리텐션 느낌 분석

In [ ]:
first_create = (
    events[events["event_name"] == "coupon__complete__create"]
    .groupby("user_id", as_index=False)["timestamp"]
    .min()
    .rename(columns={"timestamp": "first_create_at"})
)

display(first_create.head())

In [ ]:
detail_views = events[events["event_name"] == "coupon__view__detail"][["user_id", "timestamp"]].copy()

retention_df = first_create.merge(detail_views, on="user_id", how="left")
retention_df = retention_df[retention_df["timestamp"] > retention_df["first_create_at"]].copy()

retention_df["days_since_create"] = (
    retention_df["timestamp"].dt.floor("D") - retention_df["first_create_at"].dt.floor("D")
).dt.days

d1_users = set(retention_df.loc[retention_df["days_since_create"] <= 1, "user_id"].unique())
d7_users = set(retention_df.loc[retention_df["days_since_create"] <= 7, "user_id"].unique())

created_users = set(first_create["user_id"].unique())

d1_retention = round(len(d1_users) / len(created_users) * 100, 2)
d7_retention = round(len(d7_users) / len(created_users) * 100, 2)

retention_summary = pd.DataFrame({
    "metric": ["created_users", "d1_return_users", "d7_return_users", "d1_retention_pct", "d7_retention_pct"],
    "value": [len(created_users), len(d1_users), len(d7_users), d1_retention, d7_retention]
})

display(retention_summary)

In [ ]:
retention_chart = pd.DataFrame({
    "period": ["D1", "D7"],
    "retention_pct": [d1_retention, d7_retention]
})

plt.figure(figsize=(5, 4))
plt.bar(retention_chart["period"], retention_chart["retention_pct"])
plt.title("간단 리텐션(D1 / D7)")
plt.xlabel("기간")
plt.ylabel("리텐션(%)")
plt.tight_layout()
plt.show()

## 8. 일별 등록 추이

In [ ]:
daily_create = (
    events[events["event_name"] == "coupon__complete__create"]
    .assign(create_date=lambda df: df["timestamp"].dt.date)
    .groupby("create_date")
    .size()
    .reset_index(name="created_coupons")
)

display(daily_create.head())

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(daily_create["create_date"], daily_create["created_coupons"], marker="o")
plt.title("일별 등록 수")
plt.xlabel("날짜")
plt.ylabel("등록 수")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 9. 핵심 인사이트 요약

In [ ]:
print("=== 쿠왕 핵심 인사이트 요약 ===")
print(f"1. 전체 사용 완료율은 {redemption_rate}% 입니다.")
print(f"2. 전체 만료율은 {expiration_rate}% 입니다.")
print(f"3. 알림 오픈 그룹의 쿠폰 사용 완료율은 "
      f"{coupon_open_effect.loc[coupon_open_effect['notification_group']=='opened', 'coupon_redemption_rate_pct'].iloc[0]:.2f}% 입니다.")
print(f"4. 미오픈 그룹의 쿠폰 사용 완료율은 "
      f"{coupon_open_effect.loc[coupon_open_effect['notification_group']=='not_opened', 'coupon_redemption_rate_pct'].iloc[0]:.2f}% 입니다.")
print(f"5. D1 리텐션은 {d1_retention}%, D7 리텐션은 {d7_retention}% 입니다.")
print("6. 만료율이 높은 브랜드를 중심으로 알림 타이밍 개선 실험을 제안할 수 있습니다.")

## 10. 다음 확장 아이디어

In [ ]:
# 이 노트북은 CSV 기준 재현 분석용입니다.
# 이후 확장 방향:
# 1. SQLite 연결 기반 분석
# 2. 웹 대시보드와 동일한 KPI 정의 적용
# 3. 알림 메시지/타이밍 A/B 설계
# 4. 사용자 세그먼트(브랜드/쿠폰유형/만료임박일) 분석 추가
print("노트북 구성이 완료되었습니다.")